In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import stable_retro
import matplotlib.pyplot as plt
import numpy as np
from gymnasium.wrappers import RecordVideo
import os
from datetime import datetime

#### Mario Environment and Rewards

In [2]:
def compute_reward(prev_info, curr_info, done):
    reward = 0.0

    # --- Milestones ---
    if curr_info.get('midpoint_flag', 0) > prev_info.get('midpoint_flag', 0):
        reward += 5.0

    if curr_info.get('game_mode', 20) == 12 and prev_info.get('game_mode', 20) == 20:
        reward += 10.0

    # --- Powerups ---
    if curr_info.get('powerup_status', 0) > prev_info.get('powerup_status', 0):
        reward += 1.0

    # reward -= 0.001

    return reward


In [3]:
ACTIONS = [
    [0,0,0,0,0,0,0,0,0,0,0,0],  # nothing
    [0,0,0,0,0,0,0,1,0,0,0,0],  # right
    [1,0,0,0,0,0,0,1,0,0,0,0],  # right + B (jump)
    [0,1,0,0,0,0,0,1,0,0,0,0],  # right + Y (run)
    [1,1,0,0,0,0,0,1,0,0,0,0],  # right + B + Y (run jump)
    [1,0,0,0,0,0,0,0,0,0,0,0],  # B (jump in place)
    [0,0,0,0,0,0,1,0,0,0,0,0],  # left
    [1,0,0,0,0,0,1,0,0,0,0,0],  # left + B (jump left)
    [0,1,0,0,0,0,1,0,0,0,0,0],  # left + Y (run)
    [0,0,0,0,1,0,0,0,0,0,0,0],  # up
    [0,0,0,0,0,1,0,0,0,0,0,0],  # down
]

class SMWEnv(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        # Override the observation space to match your tuple state
        self.actions = ACTIONS
        self.observation_space = gym.spaces.Box(
            low=np.array([0, 0, 0, 0, -128, 0], dtype=np.float32),
            high=np.array([255, 9999, 255, 999, 128, 1], dtype=np.float32),
            dtype=np.float32
        )
        self.action_space = gym.spaces.Discrete(len(ACTIONS))
        self.stuck_counter = 0
        self.last_x = 0
        self.max_x_reached = 0
        self.prev_info = {}

    def reset(self, **kwargs):
        _, info = self.env.reset(**kwargs)
        self.prev_info = {'x': 0, 'midpoint_flag': 0, 'lives': 4, 'game_mode': 20}
        self.max_x_reached = 0
        obs = self.get_obs(info)
        return obs, info

    def step(self, action):
        actual_action = self.actions[action]
        _, _, terminated, truncated, info = self.env.step(actual_action)
        obs = self.get_obs(info)
        reward = compute_reward(self.prev_info, info, terminated or truncated)
        curr_x = info.get('x', 0)
        curr_pos = info.get('screen', 0) * 256 + curr_x
        if curr_pos > self.max_x_reached:
            reward += (curr_pos - self.max_x_reached) * 0.02
            self.max_x_reached = curr_pos

        curr_time = info.get('timer_h', 0) * 100 + info.get('timer_t', 0) * 10 + info.get('timer_o', 0)
        if curr_time == 0:
            terminated = True
            reward -= 3.0
        if info.get('game_mode', 20) == 11 and self.prev_info.get('game_mode', 20) == 20:
            terminated = True
            reward -= 3.0
        if info.get('game_mode', 20) == 12 and self.prev_info.get('game_mode', 20) == 20:
            terminated = True


        if curr_x == self.last_x:
            self.stuck_counter += 1
        else:
            self.stuck_counter = 0
            self.last_x = curr_x

        if self.stuck_counter > 600:
            terminated = True
            reward -= 4.5
            self.stuck_counter = 0

        self.prev_info = info
        return obs, reward, terminated, truncated, info

    def get_obs(self, info):
        curr_time = info.get('timer_h', 0) * 100 + info.get('timer_t', 0) * 10 + info.get('timer_o', 0)
        mario_x = info.get('x', 0)
        mario_x_local = mario_x % 256
        if info.get('sprite1_x', 0) == 0:
            sprite1_dist = 0.0  # no sprite, neutral signal
        else:
            sprite1_dist = float(np.clip(info.get('sprite1_x', 0) - mario_x_local, -128, 128))
        return np.array([
            info.get('screen', 0),
            mario_x,
            info.get('y', 0),
            curr_time,
            sprite1_dist,
            info.get('midpoint_flag', 0),
        ], dtype=np.float32)

#### Actor and Critic

In [4]:
class Actor(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super().__init__()
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        self.fc1 = nn.Linear(obs_dim, 64)   # 5 inputs → 64 neurons
        self.fc2 = nn.Linear(64, 64)        # 64 → 64
        self.fc3 = nn.Linear(64, action_dim) # 64 → one output per action

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        probs = F.softmax(self.fc3(x), dim=-1)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action, log_prob

class Critic(nn.Module):
    def __init__(self, obs_dim):
        super().__init__()
        self.obs_dim = obs_dim
        self.fc1 = nn.Linear(obs_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 1)
    
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        V = self.fc3(x)
        return V

#### PPO Algorithm

In [18]:
try:
    base_env.close()
except:
    pass

In [ ]:
base_env = stable_retro.make('SuperMarioWorld-Snes-v0', render_mode=None)
# base_env = stable_retro.make('SuperMarioWorld-Snes-v0', render_mode='human')
env = SMWEnv(base_env)

total_rewards = []
steps_per_ep  = []
max_x_per_ep  = []
actor_losses  = []
critic_losses = []

In [7]:
class SMW_PPO:
    def __init__(self, env, obs_dim, action_dim, theta, n_workers, epsilon = 0.2, alpha = 0.95, gamma=0.99, num_steps=4096):
        self.env = env
        self.gamma = gamma
        self.alpha = alpha
        self.epsilon = epsilon
        self.num_steps = num_steps
        self.n_workers = n_workers
        self.num_eps = 2000

        #create array for collecting trajectories
        self.states    = []
        self.actions   = []
        self.rewards   = []
        self.log_probs = []
        self.values    = []
        self.dones     = []
        self.theta = theta

        self.state, _ = self.env.reset()

        #call in my actor and critic class and optimize them
        self.actor = Actor(obs_dim=obs_dim, action_dim=self.env.action_space.n)
        self.critic = Critic(obs_dim=obs_dim)
        self.actor_optimizer  = optim.Adam(self.actor.parameters(),  lr=1e-4)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=1e-3)

        self.policy = []
        
    def collect_trajectories(self):
        #iterate through my steps typically PPO is 2048
        for _ in range(self.num_steps):
            
            #set up my actor and critic
            action, log_prob = self.actor(torch.tensor(self.state, dtype=torch.float32))
            action_idx = action.item()
            next_state, reward, terminated, truncated, info = self.env.step(action_idx)
            done = terminated or truncated
            value = self.critic(torch.tensor(self.state, dtype=torch.float32))


            #grab all my trajectories and append them
            self.states.append(self.state)
            self.actions.append(action_idx)
            self.rewards.append(reward)
            self.log_probs.append(log_prob)
            self.values.append(value)
            self.dones.append(done)

            # reset if episode ended
            if done:
                self.state, _ = self.env.reset()
            else:
                self.state = next_state

        #return my set of trajectories
        return self.states, self.actions, self.rewards, self.log_probs, self.values, self.dones

    def r_to_go(self):
        Rt = []
        discounted_sum = 0
        
        #grab all rewards and dones and add them into my discounted sum
        for reward, done in zip(reversed(self.rewards), reversed(self.dones)):
            if done:
                discounted_sum = 0

            discounted_sum = reward + self.gamma * discounted_sum
            Rt.insert(0, discounted_sum)

        return Rt

    # def advantage_est(self):
    #     gae = torch.GAE(self.n_workers, self.num_steps, self.gamma, self.alpha)
    #     return gae

    def advantage_est(self):
        #take a generic advantage estimation
        Rt = torch.tensor(self.r_to_go(), dtype=torch.float32)
        V  = torch.stack(self.values).squeeze()
        advantages = Rt - V.detach()
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return advantages
    
    def update(self):
        states_t = torch.tensor(np.array(self.states),    dtype=torch.float32)
        
        # get new log_probs and values from current networks
        _, new_log_probs = self.actor(states_t)
        new_values = self.critic(states_t).squeeze()

        # compute the probability ratio π_new / π_old
        old_logs = torch.stack(self.log_probs).detach()
        ratio = torch.exp(new_log_probs - old_logs)

        # compute advantages
        advantages = self.advantage_est().detach()
        advantages = torch.clamp(advantages, -5, 5)

        #two candidates for the loss
        unclipped = ratio * advantages
        clipped = torch.clamp(ratio, 1-self.epsilon, 1+self.epsilon) * advantages

        # take the minimum (pessimistic bound)
        actor_loss = -torch.mean(torch.min(unclipped, clipped))
        
        # critic just minimizes prediction error
        critic_loss = nn.MSELoss()(new_values, torch.tensor(self.r_to_go(), dtype=torch.float32))

        # backprop actor
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.actor.parameters(), 0.2)
        self.actor_optimizer.step()

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.critic.parameters(), 0.2)
        self.critic_optimizer.step()

        return actor_loss.item(), critic_loss.item()
        
    def clear_buffer(self):
        self.states    = []
        self.actions   = []
        self.rewards   = []
        self.log_probs = []
        self.values    = []
        self.dones     = []

    def rollout(self):
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        checkpoint_dir = f'Checkpoints/run_{timestamp}'
        os.makedirs(checkpoint_dir, exist_ok=True)
        for ep in range(self.num_eps):
            self.state, _ = self.env.reset()  # always start fresh
            self.collect_trajectories()

            total_rewards.append(sum(self.rewards))
            steps_per_ep.append(len(self.rewards))
            max_x_per_ep.append(max(s[0] * 256 + s[1] for s in self.states))
            actor_loss, critic_loss = self.update()
            actor_losses.append(actor_loss)
            critic_losses.append(critic_loss)

            self.clear_buffer()
            if (ep + 1) % 50 == 0:
                torch.save({
                    'actor': self.actor.state_dict(),
                    'critic': self.critic.state_dict(),
                }, f'{checkpoint_dir}/checkpoint_ep{ep+1}.pth')
            print(f"Episode {ep+1}/{self.num_eps} | Reward: {total_rewards[-1]:.2f} | Max X: {max_x_per_ep[-1]}")

In [ ]:
agent = SMW_PPO(env=env, obs_dim=6, action_dim=env.action_space.n, theta=None, n_workers=1)
agent.rollout()

Episode 1/2000 | Reward: 10.16 | Max X: 273.0
Episode 2/2000 | Reward: 10.18 | Max X: 273.0
Episode 3/2000 | Reward: 11.12 | Max X: 273.0
Episode 4/2000 | Reward: 11.08 | Max X: 272.0
Episode 5/2000 | Reward: 10.22 | Max X: 274.0
Episode 6/2000 | Reward: 10.16 | Max X: 273.0
Episode 7/2000 | Reward: 6.38 | Max X: 318.0
Episode 8/2000 | Reward: 6.90 | Max X: 345.0
Episode 9/2000 | Reward: 7.12 | Max X: 356.0
Episode 10/2000 | Reward: 8.68 | Max X: 434.0
Episode 11/2000 | Reward: 16.10 | Max X: 803.0
Episode 12/2000 | Reward: 12.52 | Max X: 624.0
Episode 13/2000 | Reward: 27.18 | Max X: 1151.0
Episode 14/2000 | Reward: 44.72 | Max X: 2116.0
Episode 15/2000 | Reward: 27.56 | Max X: 1133.0
Episode 16/2000 | Reward: 28.48 | Max X: 1184.0
Episode 17/2000 | Reward: 44.96 | Max X: 2116.0
Episode 18/2000 | Reward: 45.04 | Max X: 2116.0
Episode 19/2000 | Reward: 26.62 | Max X: 1123.0
Episode 20/2000 | Reward: 42.30 | Max X: 2115.0
Episode 21/2000 | Reward: 44.20 | Max X: 2115.0
Episode 22/2000 |

In [ ]:
import matplotlib.gridspec as gridspec

num_eps = len(total_rewards)
eps = range(1, num_eps + 1)
window = 50

def rolling_avg(data, w):
    return np.convolve(data, np.ones(w)/w, mode='valid')

fig = plt.figure(figsize=(12, 8))
fig.suptitle('SMW PPO Training', fontsize=14)
gs = gridspec.GridSpec(2, 4)

ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(eps, total_rewards, alpha=0.3, label='Raw')
ax1.plot(range(window, num_eps + 1), rolling_avg(total_rewards, window), label=f'Avg ({window}ep)')
ax1.set_title('Total Reward per Episode')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Total Reward')
ax1.legend()

ax2 = fig.add_subplot(gs[0, 2:])
ax2.plot(eps, max_x_per_ep, alpha=0.3, label='Raw')
ax2.plot(range(window, num_eps + 1), rolling_avg(max_x_per_ep, window), label=f'Avg ({window}ep)')
ax2.set_title('Max X Reached per Episode')
ax2.set_xlabel('Episode')
ax2.set_ylabel('X Position')
ax2.legend()

ax3 = fig.add_subplot(gs[1, 1:3])
ax3.plot(eps, actor_losses, alpha=0.3, label='Raw')
ax3.plot(range(window, num_eps + 1), rolling_avg(actor_losses, window), label=f'Avg ({window}ep)')
ax3.set_title('Actor Loss per Episode')
ax3.set_xlabel('Episode')
ax3.set_ylabel('Loss')
ax3.legend()

plt.tight_layout()
plt.savefig('Graphs/april4_results.png')
plt.show()


In [ ]:
actor = Actor(obs_dim=6, action_dim=env.action_space.n)

checkpoint = torch.load('Checkpoints/run_20260404_123729/checkpoint_ep1600.pth')
actor.load_state_dict(checkpoint['actor'])
actor.eval()

best_max_x = 0
for attempt in range(20):
    base_env = stable_retro.make('SuperMarioWorld-Snes-v0', render_mode='rgb_array')
    base_env = RecordVideo(base_env, video_folder='./Videos', 
                       episode_trigger=lambda x: True,
                       name_prefix=f'attempt_{attempt}')
    env = SMWEnv(base_env)

    obs, info = env.reset()
    done = False
    max_x = 0
    while not done:
        with torch.no_grad():
            action, _ = actor(torch.tensor(obs, dtype=torch.float32).unsqueeze(0))
        obs, reward, terminated, truncated, info = env.step(action.item())
        max_x = max(max_x, info.get('screen', 0) * 256 + info.get('x', 0))
        done = terminated or truncated
    env.close()
    print(f"Attempt {attempt+1} | Max X: {max_x}")
    if max_x > best_max_x:
        best_max_x = max_x


Attempt 1 | Max X: 5279
Attempt 2 | Max X: 5281
Attempt 3 | Max X: 1159
Attempt 4 | Max X: 2711
Attempt 5 | Max X: 4876
Attempt 6 | Max X: 4877
Attempt 7 | Max X: 2717
Attempt 8 | Max X: 1162
Attempt 9 | Max X: 4877
Attempt 10 | Max X: 2716
Attempt 11 | Max X: 5281
Attempt 12 | Max X: 5377
Attempt 13 | Max X: 5281
Attempt 14 | Max X: 5281
Attempt 15 | Max X: 2068
Attempt 16 | Max X: 1161
Attempt 17 | Max X: 4878
Attempt 18 | Max X: 5281
Attempt 19 | Max X: 5280
Attempt 20 | Max X: 4878


In [13]:
# test a bunch of addresses around where x should be
import stable_retro

env = stable_retro.make(game='SuperMarioWorld-Snes-v0', render_mode='human')
env.reset()

for _ in range(100):
    action = [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]  # hold right
    env.step(action)

ram = env.get_ram()
# print addresses in the range we care about
for addr in [0x00D1, 0x00D3, 0x0086, 0x00D4, 0x00B6, 0x00B8]:
    print(f"0x{addr:04X}: {ram[addr]}")
    print(0x7E0000 + 0x00D1)  # x
    print(0x7E0000 + 0x00D4)  # screen  

env.close()

RuntimeError: Cannot create multiple emulator instances per process, make sure to call env.close() on each environment before creating a new one